In [28]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# from pydantic import BaseModel, Field
# from typing import Literal
from langchain_core.tools import tool
import json
load_dotenv()

True

In [9]:
with open("bajaj_db.json") as f:
    db = json.load(f)

In [14]:
db['loans']['BFL2024001']

{'customer_name': 'Rahul Tiwari',
 'loan_type': 'Personal Loan',
 'principal': 500000,
 'emi': 8450,
 'tenure_months': 72,
 'paid_months': 50,
 'remaining_months': 22,
 'outstanding': 185900,
 'interest_rate': 11.5,
 'next_due_date': '2026-05-05',
 'prepayment_allowed': True,
 'prepayment_charge_pct': 2.0}

In [190]:
@tool
def get_details(loan_id):
    "this function will help you to get the loan details  args : loan_id (str): The ID of the loan"
    details = db['loans'].get(loan_id, 'No info for this id is available')
    return details

@tool
def next_emi_date(loan_id):
    """
    This function will help you to get the next EMI date
    args : loan_id (str): The ID of the loan
    """
    details = db['loans'].get(loan_id, {})
    return details.get('next_due_date')

@tool
def get_weather_info(city):
    "This function will return the temp of provided cities"
    if city.lower() == 'mumbai':
        return "34 degree"
    return 'We dont have api acces'

@tool
def get_policy_data(policy_name):
    """This function will help you to get the policy data
    args : policy_name (str): The name of the policy"""
    if policy_name.lower() == 'personal loan':
        return "Personal loan policy data"

# get_details('BFL2024001')
model = ChatOpenAI(model="gpt-4o-mini")
tools = [get_details, next_emi_date,get_weather_info,get_policy_data]
model_with_tool = model.bind_tools(tools)

In [191]:
prompt= ChatPromptTemplate.from_messages([
   ('system',''' Your are a expert Financial advisor.'''),
   ('human','{questions}' )])

In [192]:

chain = prompt | model_with_tool 

In [200]:
result = chain.invoke({'questions': 'what is my emi amount for loan id BFL2024001'})

In [201]:
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 191, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8c60972de7', 'id': 'chatcmpl-EKm7J3ViUPQc5oXoP4qFsz6fbfNLU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07212-ca2d-7c43-abe1-518376355163-0', tool_calls=[{'name': 'get_details', 'args': {'loan_id': 'BFL2024001'}, 'id': 'call_yEUrXZdYNzQ7mCxQOWgqlDlT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 191, 'output_tokens': 19, 'total_tokens': 210, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [202]:
result.tool_calls

[{'name': 'get_details',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_yEUrXZdYNzQ7mCxQOWgqlDlT',
  'type': 'tool_call'}]

In [203]:
tool_map = {
    "get_details": get_details,
    "next_emi_date": next_emi_date,
    "get_weather_info": get_weather_info,
    "get_policy_data": get_policy_data
}


for tc in result.tool_calls:
    tools_name = tc['name']
    print(f"tool name: {tools_name}")
    tools_args = tc['args']
    tool_id = tc['id']
    result = tool_map[tools_name].invoke(tools_args)
    print(f"Result: {result}")


tool name: get_details
Result: {'customer_name': 'Rahul Tiwari', 'loan_type': 'Personal Loan', 'principal': 500000, 'emi': 8450, 'tenure_months': 72, 'paid_months': 50, 'remaining_months': 22, 'outstanding': 185900, 'interest_rate': 11.5, 'next_due_date': '2026-05-05', 'prepayment_allowed': True, 'prepayment_charge_pct': 2.0}


In [149]:
print(result.tool_calls[0]['name'])
print(result.tool_calls[0]['args'])
print(result.tool_calls[0]['id'])

next_emi_date
{'loan_id': 'BFL2024001'}
call_gNk6aQqP92WLLcDnpIKTHBUc


In [ ]:
next_emi_date

In [152]:
result.tool_calls[0]['name']

'next_emi_date'

In [153]:
next_emi_date.invoke(result.tool_calls[0]['args'])

'2026-05-05'

In [ ]:
result.tool_calls[0]['name']

[{'name': 'next_emi_date',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_gNk6aQqP92WLLcDnpIKTHBUc',
  'type': 'tool_call'}]

In [154]:
tool_map = {
    "get_details": get_details,
    "next_emi_date": next_emi_date,
    "get_weather_info": get_weather_info
}


tool_map[result.tool_calls[0]['name']]

StructuredTool(name='next_emi_date', description='This function will help you to get the next EMI date\nargs : loan_id (str): The ID of the loan', args_schema=<class 'langchain_core.utils.pydantic.next_emi_date'>, func=<function next_emi_date at 0x123f3e020>)

In [146]:
next_emi_date.invoke({'loan_id': 'BFL2024001'})

'2026-05-05'

In [ ]:
# get_details.invoke({'loan_id': 'BFL2024001'})

{'customer_name': 'Rahul Tiwari',
 'loan_type': 'Personal Loan',
 'principal': 500000,
 'emi': 8450,
 'tenure_months': 72,
 'paid_months': 50,
 'remaining_months': 22,
 'outstanding': 185900,
 'interest_rate': 11.5,
 'next_due_date': '2026-05-05',
 'prepayment_allowed': True,
 'prepayment_charge_pct': 2.0}

In [ ]:
tool_map = {
    "get_details": get_details,
    "next_emi_date": next_emi_date,
    "get_weather_info": get_weather_info
}


for tc in result.tool_calls:
    tools_name = tc['name']
    print(f"tool name: {tools_name}")
    tools_args = tc['args']
    tool_id = tc['id']
    result = tool_map[tools_name].invoke(tools_args)
    print(f"Result: {result}")


tool name: get_details
Result: {'customer_name': 'Rahul Tiwari', 'loan_type': 'Personal Loan', 'principal': 500000, 'emi': 8450, 'tenure_months': 72, 'paid_months': 50, 'remaining_months': 22, 'outstanding': 185900, 'interest_rate': 11.5, 'next_due_date': '2026-05-05', 'prepayment_allowed': True, 'prepayment_charge_pct': 2.0}


## Experiments

In [86]:
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 143, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4abd59899c', 'id': 'chatcmpl-EKlZb3mi4C2YFOg0e6Na1NRQwUWdn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a071f2-e367-7e21-8def-c0623dadb2e1-0', tool_calls=[{'name': 'get_weather_info', 'args': {'city': 'Mumbai'}, 'id': 'call_bdij8sAnf0P5eh9j66AZr2g5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 15, 'total_tokens': 158, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio'

In [87]:
result.tool_calls

[{'name': 'get_weather_info',
  'args': {'city': 'Mumbai'},
  'id': 'call_bdij8sAnf0P5eh9j66AZr2g5',
  'type': 'tool_call'}]

In [42]:
result.tool_calls

[{'name': 'get_details',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_vMxr1ds6mQ7oxW2iQ7mSxzpb',
  'type': 'tool_call'}]